# Phase 3 — Panel Regression (Fixed Effects)

**Goal:** Test whether nurse staffing is statistically significantly associated with patient outcomes, controlling for GDP, health expenditure, and hospital beds.

Method: Fixed Effects panel regression (within estimator) using `linearmodels`.

Model: `outcome_it = β × nurses_it + γ × controls_it + α_i + ε_it`
- `i` = country, `t` = year
- `α_i` = country fixed effect (absorbs time-invariant differences)

Outcomes tested:
1. 30-day AMI mortality
2. 30-day stroke mortality
3. Average length of stay

In [1]:
import pandas as pd
import numpy as np
from linearmodels.panel import PanelOLS
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

PROCESSED = '../data/processed/'
OUTPUTS = '../data/outputs/'

df = pd.read_csv(PROCESSED + 'master_dataset.csv')
df = df.dropna(subset=['nurses_per_10k','gdp_per_capita','health_exp_pct_gdp','beds_per_1000'])
df = df.set_index(['country', 'year'])
print('Panel shape:', df.shape)

Panel shape: (540, 12)


## 1. Model: Staffing → AMI Mortality

In [2]:
ami_data = df.dropna(subset=['mortality_ami_30d'])

model_ami = PanelOLS(
    dependent=ami_data['mortality_ami_30d'],
    exog=sm.add_constant(ami_data[['nurses_per_10k','gdp_per_capita','health_exp_pct_gdp','beds_per_1000']]),
    entity_effects=True,
    time_effects=True
)
res_ami = model_ami.fit(cov_type='clustered', cluster_entity=True)
print(res_ami.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:      mortality_ami_30d   R-squared:                        0.1148
Estimator:                   PanelOLS   R-squared (Between):             -1.2442
No. Observations:                 304   R-squared (Within):              -0.4612
Date:                Mon, May 18 2026   R-squared (Overall):             -0.9175
Time:                        20:30:37   Log-likelihood                   -401.12
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      8.3040
Entities:                          23   P-value                           0.0000
Avg Obs:                       13.217   Distribution:                   F(4,256)
Min Obs:                       1.0000                                           
Max Obs:                       21.000   F-statistic (robust):             0.7030
                            

## 2. Model: Staffing → Stroke Mortality

In [3]:
stroke_data = df.dropna(subset=['mortality_stroke_30d'])

model_stroke = PanelOLS(
    dependent=stroke_data['mortality_stroke_30d'],
    exog=sm.add_constant(stroke_data[['nurses_per_10k','gdp_per_capita','health_exp_pct_gdp','beds_per_1000']]),
    entity_effects=True,
    time_effects=True
)
res_stroke = model_stroke.fit(cov_type='clustered', cluster_entity=True)
print(res_stroke.summary)

                           PanelOLS Estimation Summary                            
Dep. Variable:     mortality_stroke_30d   R-squared:                        0.0462
Estimator:                     PanelOLS   R-squared (Between):             -0.0469
No. Observations:                   291   R-squared (Within):               0.0626
Date:                  Mon, May 18 2026   R-squared (Overall):             -0.0240
Time:                          20:30:37   Log-likelihood                   -587.28
Cov. Estimator:               Clustered                                           
                                          F-statistic:                      2.9399
Entities:                            23   P-value                           0.0212
Avg Obs:                         12.652   Distribution:                   F(4,243)
Min Obs:                         1.0000                                           
Max Obs:                         21.000   F-statistic (robust):             1.4531
    

## 3. Model: Staffing → Length of Stay

In [4]:
los_data = df.dropna(subset=['avg_length_of_stay'])

model_los = PanelOLS(
    dependent=los_data['avg_length_of_stay'],
    exog=sm.add_constant(los_data[['nurses_per_10k','gdp_per_capita','health_exp_pct_gdp','beds_per_1000']]),
    entity_effects=True,
    time_effects=True
)
res_los = model_los.fit(cov_type='clustered', cluster_entity=True)
print(res_los.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:     avg_length_of_stay   R-squared:                        0.0883
Estimator:                   PanelOLS   R-squared (Between):             -0.2285
No. Observations:                 459   R-squared (Within):               0.3958
Date:                Mon, May 18 2026   R-squared (Overall):             -0.0734
Time:                        20:30:38   Log-likelihood                   -393.44
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      9.8802
Entities:                          26   P-value                           0.0000
Avg Obs:                       17.654   Distribution:                   F(4,408)
Min Obs:                       1.0000                                           
Max Obs:                       22.000   F-statistic (robust):             0.7555
                            

## 4. Results Summary Table

In [5]:
results = []
for name, res in [('AMI Mortality', res_ami), ('Stroke Mortality', res_stroke), ('Length of Stay', res_los)]:
    coef = res.params['nurses_per_10k']
    pval = res.pvalues['nurses_per_10k']
    ci_low, ci_high = res.conf_int().loc['nurses_per_10k']
    results.append({'Outcome': name, 'Coefficient': round(coef, 4), 'p-value': round(pval, 4), '95% CI Low': round(ci_low, 4), '95% CI High': round(ci_high, 4), 'Significant': pval < 0.05})

results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUTS + 'regression_results.csv', index=False)
print(results_df.to_string(index=False))

         Outcome  Coefficient  p-value  95% CI Low  95% CI High  Significant
   AMI Mortality       0.0179   0.4280     -0.0264       0.0622        False
Stroke Mortality      -0.0612   0.2692     -0.1701       0.0477        False
  Length of Stay      -0.0147   0.3435     -0.0451       0.0157        False
